In [28]:
##### Cleans India labor data
# changes units and reformats

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling
import rasterstats
from rasterstats import zonal_stats

In [29]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
labor = pd.read_csv(f"{cd}/Data/Raw/Sub_National/India/ICRISAT-District Level Data.csv")

IND_codes = pd.read_csv(f"{cd}/Data/Correspondence_tables/IND_districts.csv")

IND_districts = gpd.read_file('/Users/carinamanitius/Documents/Data/Admin_Boundaries/India/India_Districts.shp')

ag_production = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# Set save path
save_path = f"{cd}/Data/Clean/Labor/IND_labor.csv"

In [30]:
##### Clean

# convert to total jobs
labor['ag_workers'] = labor['AGRICULTURAL WORKERS POPULATION (1000 Number)'] * 1000

# combine with ID's
labor = labor.merge(IND_codes, left_on=['Dist Code', 'State Code'], right_on=['DIST_CODE', 'STATE_CODE'], how='inner')

# reformat to wide 
labor_wide = labor.pivot(
    index='GID_2',
    columns='Year',
    values='ag_workers'  
).reset_index()

# add units
labor_wide['Units'] = 'Ag labor - jobs'

# remerge with state and district ID
labor_wide = labor_wide.merge(IND_codes, on='GID_2', how='inner')

# convert type
labor_wide['STATE_CODE'] = labor_wide['STATE_CODE'].astype('Int64')
labor_wide['DIST_CODE'] = labor_wide['DIST_CODE'].astype('Int64')

# reorder columns
columns_to_keep = ['GID_2', 'STATE_CODE', 'DIST_CODE', 'Units', 2001, 2011]
labor_wide = labor_wide[columns_to_keep]

In [31]:
### Final cleaning 

labor_wide['ISO3'] = 'IND'
labor_wide['Year'] = 2011
labor_wide['GEO_ID_NAME'] = 'GID_2'

labor_wide = labor_wide.rename(columns={2011: 'ag_labor_jobs', 'GID_2': 'GEO_ID'})

col_to_keep = ['ISO3', 'GEO_ID', 'GEO_ID_NAME', 'Year', 'ag_labor_jobs']
labor_wide = labor_wide[col_to_keep]

In [32]:
##### Fill missing state data (LABOR)

### 1: identify missing states 
IND_districts = IND_districts.merge(labor_wide, left_on='GID_2', right_on='GEO_ID', how='left')

IND_districts_missing = IND_districts[IND_districts['ag_labor_jobs'].isna()].copy()
IND_districts_filled  = IND_districts[IND_districts['ag_labor_jobs'].notna()].copy()

### 2: calculate the total ag production in each state
def subnational_production_stats(geo, raster_path, column_name, nodata=-9999):

    with rasterio.open(raster_path) as src:
        raster_crs = src.crs

    # ensure crs match
    gdf_proj = geo.to_crs(raster_crs)

    zonal = rasterstats.zonal_stats(
        gdf_proj,
        raster_path,
        stats="sum",
        nodata=nodata
    )

    geo[column_name] = [z["sum"] for z in zonal]
    return geo

IND_districts_missing_prod = subnational_production_stats(IND_districts_missing, ag_production, "total_production_USD")
IND_districts_filled_prod = subnational_production_stats(IND_districts_filled, ag_production, "total_production_USD")

### 3: Get the average machinery per $ for filled data
IND_districts_filled_prod = IND_districts_filled_prod.dropna()
total_jobs = IND_districts_filled_prod['ag_labor_jobs'].sum()
production_total_USD = IND_districts_filled_prod['total_production_USD'].sum()

jobs_per_USD = total_jobs / production_total_USD

### 4: use machinery per $ to fill missing data
IND_districts_missing_prod['ag_labor_jobs'] = jobs_per_USD * IND_districts_missing_prod['total_production_USD']

### 5: recombine
IND_districts = pd.concat([IND_districts_missing_prod, IND_districts_filled_prod])

In [33]:
### Clean back to final format 
col_to_keep = ['ISO3', 'GID_2', 'GEO_ID_NAME', 'Year', 'ag_labor_jobs']
IND_districts = IND_districts[col_to_keep]

IND_districts['ag_labor_jobs'] = IND_districts['ag_labor_jobs'].fillna(0)
IND_districts['ISO3'] = IND_districts['ISO3'].fillna('IND')
IND_districts['GEO_ID_NAME'] = IND_districts['GEO_ID_NAME'].fillna('GID_2')
IND_districts['Year'] = IND_districts['Year'].fillna(2011)

IND_districts = IND_districts.rename(columns={'GID_2': 'GEO_ID'})

In [34]:
##### Save cleaned data
IND_districts.to_csv(save_path, index=False)